# Modelo de predicción de no afiliación a salud

En este notebook entrenamos un modelo supervisado que predice si una persona de la encuesta ASP 2022 **no tiene afiliación al sistema de salud**, a partir de sus características sociodemográficas y de la localidad donde vive. Los pasos son:

- Preparar la variable objetivo y las variables de entrada
- Entrenar una regresión logística como línea base
- Entrenar modelos de ensamble (Random Forest y Gradient Boosting)
- Optimizar hiperparámetros con validación cruzada
- Evaluar el mejor modelo con datos que no vio
- Interpretar el modelo con SHAP
- Guardar el modelo para el endpoint de AWS

In [1]:
#importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split

In [2]:
#cargamos el dataset limpio (el mismo que usamos para el clustering)
df = pd.read_csv("../data/processed/asp_2022_variables_limpio.csv")
print("Filas y columnas:", df.shape)

df["afiliado_seg_social_salud"].value_counts()

Filas y columnas: (3003, 50)


afiliado_seg_social_salud
Sí                     1499
No                     1492
No sabe/No responde      12
Name: count, dtype: int64

# Variable objetivo

La propuesta planteaba tres clases (afiliado / no afiliado / desconocido), pero "No sabe/No responde" son solo 12 personas (0,4 %): no alcanza para que un modelo aprenda esa clase. Por eso la dejamos **binaria**:

- **1 = NO afiliada** (es lo que queremos detectar)
- **0 = afiliada**

y quitamos las 12 personas sin respuesta. Las clases quedan **balanceadas** (casi 50 % y 50 %), así que no hace falta remuestreo.

In [3]:
#quitamos a quienes no saben o no respondieron
df = df[df["afiliado_seg_social_salud"] != "No sabe/No responde"].reset_index(drop=True)

#variable objetivo: 1 = NO afiliada, 0 = afiliada
y = (df["afiliado_seg_social_salud"] == "No").astype(int)

print("Personas:", len(df))
print("Proporción de NO afiliadas: %0.3f" % y.mean())

Personas: 2991
Proporción de NO afiliadas: 0.499


# Variables de entrada

Usamos **las mismas 18 variables del clustering** con la misma transformación, para que el modelo y los perfiles hablen el mismo idioma. Para no copiar los diccionarios a mano, los leemos del archivo `clustering_config.json` que guardamos en el notebook de minería.

Además agregamos **3 indicadores de la localidad** (de SaluData) que pide la propuesta: pobreza monetaria 2021, coeficiente de Gini 2021 y % de afiliados al régimen subsidiado en 2022.

**Variables que NO entran (fuga de información):** `regimen_salud`, las preguntas de barreras en salud, `atencion_medica_ult_anio` y similares. Solo se les preguntaron a quienes están afiliadas o dependen de estar afiliada: si las usamos, el modelo "haría trampa" con información que ya revela la respuesta.

**Tampoco usamos `cluster` como variable:** se calcula con las mismas 18 variables, así que no aporta información nueva.

In [4]:
#diccionarios de recodificación guardados en el notebook de minería
with open("modelos/clustering_config.json", encoding="utf-8") as archivo:
    config = json.load(archivo)

variables = config["variables"]   #las 18 variables, en el mismo orden del clustering
print(variables)

['edad', 'personas_hogar', 'dependientes_economicos', 'autopercepcion_nivel_economico', 'horas_semana_asp', 'edad_inicio_asp', 'tiempo_viviendo_bogota', 'nivel_educativo', 'tipo_vivienda', 'migrante', 'grupo_etnico', 'victima_conflicto_armado', 'ingresos_insuficientes', 'paga_intermediario_tercero', 'discriminacion_general', 'amenazas_recibidas', 'forzada_relaciones_sexuales', 'violencia_fuerza_publica']


In [5]:
#tomamos las columnas originales del CSV
columnas_originales = ["edad", "personas_hogar", "dependientes_economicos", "autopercepcion_nivel_economico",
                       "horas_semana_asp", "edad_inicio_asp", "tiempo_viviendo_bogota", "nivel_educativo",
                       "tipo_vivienda", "pais_nacimiento", "autoreconocimiento_etnico", "victima_conflicto_armado",
                       "ingresos_suficientes_gastos", "paga_intermediario_tercero", "discriminacion_general",
                       "amenazas_recibidas", "forzada_relaciones_sexuales", "violencia_fuerza_publica"]
X = df[columnas_originales].copy()

#"No sabe/No responde" -> moda (igual que en minería)
for col in X.columns:
    if X[col].dtype != "int64" and X[col].dtype != "float64":
        moda = X[col][X[col] != "No sabe/No responde"].mode()[0]
        X[col] = X[col].replace("No sabe/No responde", moda)

#ordinales con los diccionarios del JSON
for col, diccionario in config["recodificacion"].items():
    X[col] = X[col].map(diccionario)

#binarias a 1 y 0
X["pais_nacimiento"] = (X["pais_nacimiento"] != "Colombia").astype(int)
X["autoreconocimiento_etnico"] = (X["autoreconocimiento_etnico"] != "Ninguno").astype(int)
X["ingresos_suficientes_gastos"] = (X["ingresos_suficientes_gastos"] == "No").astype(int)
for col in ["victima_conflicto_armado", "paga_intermediario_tercero", "discriminacion_general",
            "amenazas_recibidas", "forzada_relaciones_sexuales", "violencia_fuerza_publica"]:
    X[col] = (X[col] == "Sí").astype(int)

X = X.rename(columns={"pais_nacimiento": "migrante",
                      "autoreconocimiento_etnico": "grupo_etnico",
                      "ingresos_suficientes_gastos": "ingresos_insuficientes"})

#comprobamos que quedaron igual que en el clustering
print("Mismas variables y orden que el clustering:", list(X.columns) == variables)

Mismas variables y orden que el clustering: True


In [6]:
#indicadores de la localidad (un valor por localidad) desde la base integrada con SaluData
indicadores = pd.read_csv("../data/processed/asp_2022_saludata_integrado.csv")
indicadores = indicadores[["localidad", "pobreza_monetaria_pct_2021", "gini_2021",
                           "subsidiado_pct_propio_2022"]].drop_duplicates()
print("Localidades con indicadores:", len(indicadores))

#pegamos los indicadores a cada persona según su localidad
X["localidad"] = df["localidad"]
X = X.merge(indicadores, on="localidad", how="left").drop(columns="localidad")

print("Valores vacíos:", X.isnull().sum().sum())
print("Filas y columnas de X:", X.shape)
X.head()

Localidades con indicadores: 19
Valores vacíos: 0
Filas y columnas de X: (2991, 21)


,edad,personas_hogar,dependientes_economicos,autopercepcion_nivel_economico,horas_semana_asp,edad_inicio_asp,tiempo_viviendo_bogota,nivel_educativo,tipo_vivienda,migrante,...,victima_conflicto_armado,ingresos_insuficientes,paga_intermediario_tercero,discriminacion_general,amenazas_recibidas,forzada_relaciones_sexuales,violencia_fuerza_publica,pobreza_monetaria_pct_2021,gini_2021,subsidiado_pct_propio_2022
0,19,1,0,5,48,18,0,4,0,0,...,0,1,1,0,0,0,0,23.94,0.46,14.39
1,26,1,2,4,48,26,1,5,1,1,...,0,1,1,0,0,0,0,23.94,0.46,14.39
2,21,7,4,5,48,21,1,2,0,1,...,0,1,1,0,1,1,1,23.94,0.46,14.39
3,20,2,0,1,42,16,0,3,2,0,...,0,0,0,0,0,0,0,23.94,0.46,14.39
4,20,1,4,1,56,16,0,2,2,1,...,0,1,0,0,0,0,0,23.94,0.46,14.39


# Separación en entrenamiento y prueba

Separamos el **20 % de las personas como conjunto de prueba** y no lo tocamos hasta el final: es como un examen con preguntas que el modelo nunca vio. Con el 80 % restante entrenamos y comparamos modelos usando validación cruzada.

Usamos `stratify=y` para que ambos conjuntos tengan la misma proporción de personas no afiliadas.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y, random_state=42)

print("Entrenamiento:", X_train.shape, " Proporción NO afiliadas: %0.3f" % y_train.mean())
print("Prueba:       ", X_test.shape, " Proporción NO afiliadas: %0.3f" % y_test.mean())

Entrenamiento: (2392, 21)  Proporción NO afiliadas: 0.499
Prueba:        (599, 21)  Proporción NO afiliadas: 0.499
